# NephroPatch — CKD Binary Classification Model Comparison

This notebook is a cleaned, reproducible version of the original model-comparison experiment.

### Evaluation changes
- The train/test split is created **before** scaling or oversampling.
- `MinMaxScaler` is fit only on the training fold.
- `SMOTE` is applied only to training data through an `imblearn` pipeline.
- The test set remains untouched until final evaluation.
- Classification reports and confusion matrices use `y_true` and `y_pred` in the correct order.
- The split is stratified to preserve the original diagnosis distribution.

The original project reported 97.61% accuracy for the binary XGBoost experiment. Because the original notebook used preprocessing/SMOTE before the split, rerun this cleaned notebook and treat the new holdout result as the reproducible evaluation.

## Imports

In [ ]:
import sys
print(sys.executable)

## Load data

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

RANDOM_STATE = 42

In [ ]:
DATA_PATH = Path("../data/Chronic_Kidney_Disease_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "See ../data/README.md for the expected dataset source and filename."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

## Prepare features and target

In [ ]:
DROP_COLUMNS = [
    "PatientID",
    "DoctorInCharge",
    "EducationLevel",
    "SerumCreatinine",
    "GFR",
    "ProteinInUrine",
    "ACR",
    "SerumElectrolytesSodium",
    "SerumElectrolytesPotassium",
    "SerumElectrolytesCalcium",
    "SerumElectrolytesPhosphorus",
    "HemoglobinLevels",
    "CholesterolTotal",
    "CholesterolLDL",
    "CholesterolHDL",
    "CholesterolTriglycerides",
    "MedicalCheckupsFrequency",
    "MedicationAdherence",
    "HealthLiteracy",
    "HbA1c",
]

missing_drop_columns = sorted(set(DROP_COLUMNS) - set(df.columns))
if missing_drop_columns:
    raise ValueError(f"Expected columns missing from dataset: {missing_drop_columns}")

model_df = df.drop(columns=DROP_COLUMNS).copy()

X = model_df.drop(columns="Diagnosis")
y = model_df["Diagnosis"]

print("Features:", X.shape[1])
print("Class counts:")
print(y.value_counts().sort_index())

## Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Define models and leakage-safe pipelines

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(random_state=RANDOM_STATE),
    "Support Vector Machine": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
    ),
}

def build_pipeline(model):
    return Pipeline(
        steps=[
            ("scaler", MinMaxScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("model", model),
        ]
    )

## Train and evaluate

In [ ]:
results = []
reports = {}
confusion_matrices = {}
trained_pipelines = {}

for name, model in models.items():
    pipeline = build_pipeline(model)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    results.append({"model": name, "accuracy": accuracy})
    reports[name] = classification_report(y_test, y_pred, digits=4)
    confusion_matrices[name] = confusion_matrix(y_test, y_pred)
    trained_pipelines[name] = pipeline

results_df = (
    pd.DataFrame(results)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

results_df["accuracy_percent"] = (results_df["accuracy"] * 100).round(2)
results_df[["model", "accuracy_percent"]]

## Visualize model comparison

In [ ]:
plot_df = results_df.sort_values("accuracy_percent")

plt.figure(figsize=(9, 5))
plt.barh(plot_df["model"], plot_df["accuracy_percent"])
plt.xlabel("Holdout accuracy (%)")
plt.title("CKD binary classification — clean holdout comparison")
plt.tight_layout()
plt.show()

## XGBoost evaluation

In [ ]:
print(reports["XGBoost"])

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrices["XGBoost"],
    display_labels=trained_pipelines["XGBoost"].classes_,
).plot()

plt.title("XGBoost — CKD vs non-CKD")
plt.tight_layout()
plt.show()

## Save the full XGBoost pipeline locally

In [ ]:
MODEL_PATH = Path("../models/ckd_binary_xgboost_pipeline.joblib")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "pipeline": trained_pipelines["XGBoost"],
        "feature_names": X.columns.tolist(),
    },
    MODEL_PATH,
)

print(f"Saved locally to: {MODEL_PATH}")
print("Note: generated .joblib files should stay out of Git and are ignored by .gitignore.")

In [ ]:
results_df[["model", "accuracy_percent"]]